In [0]:
# Configuration

CATALOG        = "adb_retailedge_dev"
SCHEMA         = "healthcare_cms"
STORAGE_ACCT   = "retailedgestorage1"
CONTAINER      = "healthcare"

BASE_PATH = f"abfss://{CONTAINER}@{STORAGE_ACCT}.dfs.core.windows.net/raw"
PROVIDERS_PATH = f"{BASE_PATH}/providers/cms_providers_2024.csv"
DRUGS_PATH     = f"{BASE_PATH}/drugs/cms_drugs_2024.csv"
INPATIENT_PATH = f"{BASE_PATH}/inpatient/cms_inpatient_2024.csv"


print(f"CATALOG: {CATALOG}")
print(f"SCHEMA: {SCHEMA}")
print(f"BASE_PATH: {BASE_PATH}")

CATALOG: adb_retailedge_dev
SCHEMA: healthcare_cms
BASE_PATH: abfss://healthcare@retailedgestorage1.dfs.core.windows.net/raw


In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
print(f"Schema ready: {CATALOG}.{SCHEMA}")

Schema ready: adb_retailedge_dev.healthcare_cms


In [0]:
# Bronze writter function

from pyspark.sql. functions import current_timestamp, lit

def write_bronze(df, table_name,source_file):
    bronze_df = df\
        .withColumn("_ingestion_timestamp", current_timestamp())\
        .withColumn("_source_file", lit(source_file))\
        .withColumn("_pipeline_name", lit("healthcare_cms_pipeline"))\
        .withColumn("_layer", lit("bronze"))
    
    bronze_df.write\
        .format("delta")\
        .mode("overwrite")\
        .option("overwriteSchema", "true")\
        .saveAsTable(f"{CATALOG}.{SCHEMA}.bronze_{table_name}")

    count = spark.table(f"{CATALOG}.{SCHEMA}.bronze_{table_name}").count()
    print(f"bronze_{table_name} : {count:,} rows written")
    
print("Bronze writer function ready")

Bronze writer function ready


In [0]:
# Load Providers (this will take a few minutes — 3GB file)
print("Reading providers CSV from ADLS...")
df_providers = spark.read.csv(PROVIDERS_PATH, header=True, inferSchema=False)
print(f"Rows read: {df_providers.count():,}")
print(f"Columns: {len(df_providers.columns)}")

print("Writting to bronze table...")
write_bronze(df_providers, "providers", "cms_providers_2024.csv")

Reading providers CSV from ADLS...
Rows read: 592,965
Columns: 28
Writting to bronze table...
bronze_providers : 592,965 rows written


In [0]:
# Load Drugs
print("Reading drugs CSV from ADLS...")
df_drugs = spark.read.csv(DRUGS_PATH, header=True, inferSchema=False)
print(f"Rows read: {df_drugs.count():,}")
print(f"Columns: {len(df_drugs.columns)}")

print("Writting to bronze table...")
write_bronze(df_drugs, "drugs", "cms_drugs_2024.csv")

Reading drugs CSV from ADLS...
Rows read: 14,536
Columns: 46
Writting to bronze table...
bronze_drugs : 14,536 rows written


In [0]:
  from pyspark.sql import SparkSession
  import subprocess

  # List files in the inpatient folder
  files = dbutils.fs.ls("abfss://healthcare@retailedgestorage1.dfs.core.windows.net/raw/inpatient/")
  for f in files:
      print(f.name, f.size)

cms_inpatient_2024.CSV 3066131


In [0]:
print("Reading inpatient CSV from ADLS...")
df_inpatient = spark.read.csv(INPATIENT_PATH, header=True, inferSchema=False)
print(f"Rows read: {df_inpatient.count():,}")
print(f"Columns  : {len(df_inpatient.columns)}")
print("Writing to bronze table...")
write_bronze(df_inpatient, "inpatient", "cms_inpatient_2024.csv")
print("Done.")

Reading inpatient CSV from ADLS...
Rows read: 26,571
Columns  : 9
Writing to bronze table...
bronze_inpatient : 26,571 rows written
Done.


In [0]:
# Bronze Verification

tables = ["providers","drugs","inpatient"]
total = 0

print("=" * 45)
print("BRONZE LAYER VERIFICATION")
print("=" * 45)

for t in tables:
    count = spark.table(f"{CATALOG}.{SCHEMA}.bronze_{t}").count()
    total += count
    print(f"bronze_{t:<12}: {count:>10,} rows")
print("-" * 45)
print(f"{'TOTAL':<18}: {total:>10,} rows")

BRONZE LAYER VERIFICATION
bronze_providers   :    592,965 rows
bronze_drugs       :     14,536 rows
bronze_inpatient   :     26,571 rows
---------------------------------------------
TOTAL             :    634,072 rows


In [0]:
  print("=== PROVIDERS SCHEMA ===")
  spark.table(f"{CATALOG}.{SCHEMA}.bronze_providers").printSchema()

  print("=== DRUGS SCHEMA ===")
  spark.table(f"{CATALOG}.{SCHEMA}.bronze_drugs").printSchema()

  print("=== INPATIENT SCHEMA ===")
  spark.table(f"{CATALOG}.{SCHEMA}.bronze_inpatient").printSchema()

=== PROVIDERS SCHEMA ===
root
 |-- Rndrng_NPI: string (nullable = true)
 |-- Rndrng_Prvdr_Last_Org_Name: string (nullable = true)
 |-- Rndrng_Prvdr_First_Name: string (nullable = true)
 |-- Rndrng_Prvdr_MI: string (nullable = true)
 |-- Rndrng_Prvdr_Crdntls: string (nullable = true)
 |-- Rndrng_Prvdr_Ent_Cd: string (nullable = true)
 |-- Rndrng_Prvdr_St1: string (nullable = true)
 |-- Rndrng_Prvdr_St2: string (nullable = true)
 |-- Rndrng_Prvdr_City: string (nullable = true)
 |-- Rndrng_Prvdr_State_Abrvtn: string (nullable = true)
 |-- Rndrng_Prvdr_State_FIPS: string (nullable = true)
 |-- Rndrng_Prvdr_Zip5: string (nullable = true)
 |-- Rndrng_Prvdr_RUCA: string (nullable = true)
 |-- Rndrng_Prvdr_RUCA_Desc: string (nullable = true)
 |-- Rndrng_Prvdr_Cntry: string (nullable = true)
 |-- Rndrng_Prvdr_Type: string (nullable = true)
 |-- Rndrng_Prvdr_Mdcr_Prtcptg_Ind: string (nullable = true)
 |-- HCPCS_Cd: string (nullable = true)
 |-- HCPCS_Desc: string (nullable = true)
 |-- HCPCS_Dru